# Customer Purchase Prediction Using Classification Algorithms

## Project Overview

This project builds a machine learning classification model to predict whether a customer will make a purchase based on their behavioral and demographic features. The analysis implements multiple classification algorithms and evaluates their performance to identify the best model for prediction.

## Objective

- Predict customer purchase probability using classification models
- Compare performance across multiple algorithms
- Identify key features influencing purchase decisions
- Provide actionable insights for business targeting

## Problem Statement

Businesses face the challenge of identifying high-intent customers to optimize marketing spend and improve conversion rates. This project develops a classification system to automatically identify customers likely to make purchases, enabling targeted marketing campaigns and resource allocation.

## Real-World Applications

- E-commerce platforms targeting high-value customers
- Marketing automation and personalization
- Customer segmentation for sales strategies
- Ad campaign optimization and budget allocation

# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# 2. Load Dataset

In [ ]:
df = pd.read_csv('data/customer_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
print("Dataset Information:")
print(df.info())

In [ ]:
print("Statistical Summary:")
print(df.describe())

In [ ]:
print(f"Missing Values:\n{df.isnull().sum()}")
print(f"\nMissing Value Percentage:\n{(df.isnull().sum() / len(df) * 100).round(2)}%")

# 3. Exploratory Data Analysis

We analyze the dataset to understand feature distributions, relationships, and target variable balance.

## Target Distribution

In [ ]:
target_col = df.columns[-1]
print(f"Target Variable: {target_col}")
print(f"\nClass Distribution:\n{df[target_col].value_counts()}")
print(f"\nClass Balance (%):\n{df[target_col].value_counts(normalize=True).mul(100).round(2)}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
df[target_col].value_counts().plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4'])
ax.set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Purchase Decision')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## Numerical Features Distribution

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numerical_cols:
    numerical_cols.remove(target_col)
print(f"Numerical features: {numerical_cols}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols[:4]):
    axes[idx].hist(df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Boxplots for Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols[:4]):
    axes[idx].boxplot(df[col], vert=True)
    axes[idx].set_title(f'Boxplot of {col}', fontweight='bold')
    axes[idx].set_ylabel(col)

plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
correlation_matrix = df.corr(numeric_only=True)
print("Correlation with Target Variable:")
print(correlation_matrix[target_col].sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Categorical Features Analysis

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if target_col in categorical_cols:
    categorical_cols.remove(target_col)
print(f"Categorical features: {categorical_cols}")

for col in categorical_cols:
    print(f"\n{col} - Value counts:")
    print(df[col].value_counts())

In [ ]:
if len(categorical_cols) > 0:
    fig, axes = plt.subplots(1, len(categorical_cols), figsize=(5 * len(categorical_cols), 4))
    if len(categorical_cols) == 1:
        axes = [axes]
    
    for idx, col in enumerate(categorical_cols):
        df[col].value_counts().plot(kind='bar', ax=axes[idx], color='steelblue')
        axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Count')
        axes[idx].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# 4. Data Preprocessing

We prepare the data for model training by handling missing values, encoding categorical variables, and scaling numerical features.

## Handling Missing Values

In [ ]:
df_clean = df.copy()

numerical_cols_with_target = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols_with_target = df.select_dtypes(include=['object']).columns.tolist()

for col in numerical_cols_with_target:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

for col in categorical_cols_with_target:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print(f"Missing values after imputation:\n{df_clean.isnull().sum()}")

## Encoding Categorical Variables

In [ ]:
label_encoders = {}
categorical_features = df_clean.select_dtypes(include=['object']).columns.tolist()

for col in categorical_features:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

## Feature-Target Separation and Scaling

In [ ]:
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names:\n{list(X.columns)}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaled features statistics:")
print(X_scaled.describe())

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set class distribution:\n{y_train.value_counts()}")
print(f"\nTest set class distribution:\n{y_test.value_counts()}")

# 5. Model Training

We train four classification algorithms: Logistic Regression, Decision Tree, Random Forest, and K-Nearest Neighbors.

## Logistic Regression

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
print("Logistic Regression model trained successfully.")

## Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_model.fit(X_train, y_train)
print("Decision Tree model trained successfully.")

## Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("Random Forest model trained successfully.")

## K-Nearest Neighbors

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
print("K-Nearest Neighbors model trained successfully.")

# 6. Model Evaluation

We evaluate each model using multiple metrics: accuracy, precision, recall, F1-score, confusion matrix, and ROC curve.

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Evaluate model performance on both training and test sets.
    """
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    metrics = {
        'Model': model_name,
        'Train Accuracy': accuracy_score(y_train, y_pred_train),
        'Test Accuracy': accuracy_score(y_test, y_pred_test),
        'Precision': precision_score(y_test, y_pred_test, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, y_pred_test, average='weighted', zero_division=0),
        'F1 Score': f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    }
    
    return metrics, y_pred_test

results = []
predictions = {}

### Logistic Regression Evaluation

In [ ]:
lr_metrics, lr_pred = evaluate_model(lr_model, X_train, X_test, y_train, y_test, 'Logistic Regression')
results.append(lr_metrics)
predictions['Logistic Regression'] = lr_pred

print(f"Logistic Regression Performance:")
print(f"  Train Accuracy: {lr_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {lr_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {lr_metrics['Precision']:.4f}")
print(f"  Recall: {lr_metrics['Recall']:.4f}")
print(f"  F1 Score: {lr_metrics['F1 Score']:.4f}")

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_pred))
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))

### Decision Tree Evaluation

In [ ]:
dt_metrics, dt_pred = evaluate_model(dt_model, X_train, X_test, y_train, y_test, 'Decision Tree')
results.append(dt_metrics)
predictions['Decision Tree'] = dt_pred

print(f"Decision Tree Performance:")
print(f"  Train Accuracy: {dt_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {dt_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {dt_metrics['Precision']:.4f}")
print(f"  Recall: {dt_metrics['Recall']:.4f}")
print(f"  F1 Score: {dt_metrics['F1 Score']:.4f}")

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, dt_pred))
print("\nClassification Report:")
print(classification_report(y_test, dt_pred))

### Random Forest Evaluation

In [ ]:
rf_metrics, rf_pred = evaluate_model(rf_model, X_train, X_test, y_train, y_test, 'Random Forest')
results.append(rf_metrics)
predictions['Random Forest'] = rf_pred

print(f"Random Forest Performance:")
print(f"  Train Accuracy: {rf_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {rf_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {rf_metrics['Precision']:.4f}")
print(f"  Recall: {rf_metrics['Recall']:.4f}")
print(f"  F1 Score: {rf_metrics['F1 Score']:.4f}")

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

### K-Nearest Neighbors Evaluation

In [ ]:
knn_metrics, knn_pred = evaluate_model(knn_model, X_train, X_test, y_train, y_test, 'KNN')
results.append(knn_metrics)
predictions['KNN'] = knn_pred

print(f"KNN Performance:")
print(f"  Train Accuracy: {knn_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {knn_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {knn_metrics['Precision']:.4f}")
print(f"  Recall: {knn_metrics['Recall']:.4f}")
print(f"  F1 Score: {knn_metrics['F1 Score']:.4f}")

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, knn_pred))
print("\nClassification Report:")
print(classification_report(y_test, knn_pred))

# 7. Model Comparison

## Comparison Table

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.round(4)
print(results_df.to_string(index=False))

## Comparison Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

metrics_to_plot = ['Test Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(results_df))
width = 0.2

for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, results_df[metric], width, label=metric)

ax.set_xlabel('Model', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Model'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Feature Importance Analysis

Tree-based models provide feature importance scores that indicate which features have the most influence on predictions.

## Random Forest Feature Importance

In [ ]:
feature_importance_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Random Forest Feature Importance:")
print(feature_importance_rf)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance_rf['Feature'], feature_importance_rf['Importance'], color='steelblue')
ax.set_xlabel('Importance Score', fontweight='bold')
ax.set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Logistic Regression Coefficients

In [ ]:
feature_coef_lr = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print("Logistic Regression Feature Coefficients:")
print(feature_coef_lr)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in feature_coef_lr['Coefficient']]
ax.barh(feature_coef_lr['Feature'], feature_coef_lr['Coefficient'], color=colors)
ax.set_xlabel('Coefficient Value', fontweight='bold')
ax.set_title('Logistic Regression Feature Coefficients', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()

# 9. Conclusion

This project successfully demonstrates the application of multiple classification algorithms to predict customer purchase behavior. The analysis included comprehensive exploratory data analysis, data preprocessing, model training, and rigorous evaluation using multiple performance metrics.

## Key Findings

- Multiple classification algorithms were trained and evaluated on the customer dataset
- Model performance was compared across accuracy, precision, recall, and F1 score
- Feature importance analysis revealed the most influential factors in purchase prediction
- The trained models can be used for customer segmentation and targeted marketing strategies

## Next Steps

1. Deploy the best performing model for production use
2. Implement model monitoring and retraining pipelines
3. Conduct A/B testing of marketing campaigns using model predictions
4. Collect feedback and iterate on model improvements
5. Explore advanced techniques such as ensemble methods or deep learning approaches

## Model Recommendations

The choice of model should be guided by business requirements:
- For interpretability, prefer Logistic Regression or Decision Tree
- For maximum accuracy, prefer Random Forest or ensemble methods
- Consider computational constraints and deployment environment when selecting the final model